# 10 — LSTM & Transformer Baseline Comparison

Adds two deep-learning baselines to the walk-forward evaluation:
1. **LSTM** — 2-layer LSTM with embedding layer for ordinal pattern sequences
2. **Transformer** — 1-layer Transformer encoder (positional encoding + self-attention)

Both use the same ordinal pattern sequence as the sole input feature,
identical walk-forward protocol (train=365, test=30, step=30), and are
pre-specified independent of ordinal classifier results — consistent
with standard forecast evaluation protocol (Diebold & Mariano 1995).

Saves: `data/dl_baseline_results.csv`, `data/dl_dm_results.csv`
Figure: `data/fig_dl_comparison.png`

In [ ]:
import pickle
import numpy as np
import pandas as pd
import scipy.stats as stats
import warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

print('PyTorch version:', torch.__version__)
DEVICE = torch.device('cpu')

# ── Constants ────────────────────────────────────────────────────────────────
COINS        = ['BTC', 'ETH', 'BNB', 'SOL', 'XRP', 'ADA', 'DOGE', 'LTC']
TRAIN_WINDOW = 365
TEST_WINDOW  = 30
STEP         = 30
SEQ_LEN      = 20      # look-back window (days) for sequence models
N_PATTERNS   = 6       # d=3 → 3! = 6 ordinal patterns
EMBED_DIM    = 8
LSTM_HIDDEN  = 32
LSTM_LAYERS  = 2
ATTN_HEADS   = 2
EPOCHS       = 20
BATCH_SIZE   = 32
LR           = 1e-3
SEED         = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Load ordinal patterns ────────────────────────────────────────────────────
with open('data/ordinal_patterns.pkl', 'rb') as f:
    op_dict = pickle.load(f)

# Encode each tuple pattern to integer 0-5
from itertools import permutations
all_perms = list(permutations(range(3)))
perm_to_idx = {p: i for i, p in enumerate(all_perms)}

def encode_patterns(pattern_list):
    """Convert list of tuples to integer array 0..5"""
    return np.array([perm_to_idx.get(tuple(int(x) for x in p), 0)
                     for p in pattern_list], dtype=np.int64)

# ── Direction labels ─────────────────────────────────────────────────────────
log_ret = pd.read_csv('data/preprocessed_log_returns.csv',
                      index_col=0, parse_dates=True)

def direction_label(arr):
    return (np.asarray(arr) > 0).astype(int)

def accuracy_score(y_true, y_pred):
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))

def f1_score(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    denom = 2*tp + fp + fn
    return float(2*tp / denom) if denom > 0 else 0.0

print('Ordinal pattern coins:', list(op_dict.keys()))
print('Pattern count per coin:', len(op_dict[list(op_dict.keys())[0]]))
print('Log returns shape:', log_ret.shape)

In [ ]:
# ── Model Definitions ────────────────────────────────────────────────────────

class LSTMClassifier(nn.Module):
    """2-layer LSTM with embedding for ordinal pattern sequences."""
    def __init__(self, n_patterns, embed_dim, hidden, n_layers, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(n_patterns, embed_dim)
        self.lstm  = nn.LSTM(embed_dim, hidden, n_layers,
                             batch_first=True, dropout=dropout if n_layers > 1 else 0.0)
        self.fc    = nn.Linear(hidden, 1)

    def forward(self, x):
        e = self.embed(x)                      # (B, T, embed_dim)
        out, (h, _) = self.lstm(e)             # h: (layers, B, hidden)
        logit = self.fc(h[-1])                 # (B, 1)
        return logit.squeeze(1)


class TransformerClassifier(nn.Module):
    """1-layer Transformer encoder with positional encoding."""
    def __init__(self, n_patterns, embed_dim, nhead, seq_len, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(n_patterns, embed_dim)
        self.pos   = nn.Embedding(seq_len, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=nhead, dim_feedforward=embed_dim*4,
            dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.fc = nn.Linear(embed_dim, 1)

    def forward(self, x):
        B, T = x.shape
        pos  = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)
        e    = self.embed(x) + self.pos(pos)   # (B, T, embed_dim)
        out  = self.transformer(e)             # (B, T, embed_dim)
        logit = self.fc(out[:, -1, :])         # use last token → (B, 1)
        return logit.squeeze(1)


def train_model(model, X_train, y_train, epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE):
    """Train with BCEWithLogitsLoss."""
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()
    ds = TensorDataset(
        torch.tensor(X_train, dtype=torch.long),
        torch.tensor(y_train, dtype=torch.float)
    )
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True)
    for _ in range(epochs):
        for xb, yb in dl:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
    return model


def predict_model(model, X_test):
    """Return binary predictions."""
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X_test, dtype=torch.long))
        return (torch.sigmoid(logits) >= 0.5).numpy().astype(int)


def make_sequences(patterns, labels, seq_len):
    """
    Build (X, y) where X[i] = patterns[i-seq_len:i], y[i] = labels[i].
    """
    X, y = [], []
    for i in range(seq_len, len(patterns)):
        X.append(patterns[i-seq_len:i])
        y.append(labels[i])
    return np.array(X, dtype=np.int64), np.array(y, dtype=np.int64)


print('Model definitions ready.')

In [ ]:
# ── Walk-forward evaluation ──────────────────────────────────────────────────

def walk_forward_dl(coin_name):
    r = log_ret[coin_name].dropna()
    r_vals = r.values
    y_all  = direction_label(r_vals)     # labels aligned to r
    patterns = encode_patterns(op_dict[coin_name])  # length = len(r) - (d-1) = len(r) - 2
    # patterns[i] corresponds to r[i+2] (d=3, τ=1 uses r[i], r[i+1], r[i+2])
    # Align: label for pattern[i] = y_all[i+2]
    # Offset: patterns start at index 2 in r
    PAT_OFFSET = 2

    n = len(r)
    results = []

    for i_start in range(TRAIN_WINDOW, n - TEST_WINDOW + 1, STEP):
        i_end = i_start + TEST_WINDOW
        test_y = y_all[i_start:i_end]
        if len(test_y) == 0:
            continue

        date_start = str(r.index[i_start])
        date_end   = str(r.index[i_end - 1])
        row_base = dict(coin=coin_name, date_start=date_start, date_end=date_end,
                        n_test=len(test_y), target='target_direction')

        # Slice patterns aligned to r indices
        # pattern index p corresponds to r index p + PAT_OFFSET
        # train range in pattern space: 0 .. i_start - PAT_OFFSET
        train_end_p = i_start - PAT_OFFSET
        test_end_p  = i_end   - PAT_OFFSET

        if train_end_p < SEQ_LEN + 1 or test_end_p > len(patterns):
            continue

        train_pat = patterns[:train_end_p]
        train_lbl = y_all[PAT_OFFSET:i_start]
        test_pat  = patterns[train_end_p:test_end_p]
        test_lbl  = y_all[i_start:i_end][:len(test_pat)]

        if len(test_pat) == 0:
            continue

        X_train, y_train = make_sequences(train_pat, train_lbl, SEQ_LEN)
        X_test,  y_test  = make_sequences(
            np.concatenate([train_pat[-SEQ_LEN:], test_pat]),
            np.concatenate([train_lbl[-SEQ_LEN:], test_lbl]),
            SEQ_LEN
        )
        # y_test should match test_lbl
        y_test = test_lbl[:len(y_test)]

        if len(X_train) < BATCH_SIZE or len(X_test) == 0:
            continue

        torch.manual_seed(i_start)  # reproducible per window

        # ── LSTM ────────────────────────────────────────────────────────────
        lstm = LSTMClassifier(N_PATTERNS, EMBED_DIM, LSTM_HIDDEN, LSTM_LAYERS)
        lstm = train_model(lstm, X_train, y_train)
        pred_lstm = predict_model(lstm, X_test)
        results.append({**row_base, 'model': 'LSTM',
                         'acc': accuracy_score(y_test, pred_lstm),
                         'f1':  f1_score(y_test, pred_lstm)})

        # ── Transformer ─────────────────────────────────────────────────────
        tfm = TransformerClassifier(N_PATTERNS, EMBED_DIM, ATTN_HEADS, SEQ_LEN)
        tfm = train_model(tfm, X_train, y_train)
        pred_tfm = predict_model(tfm, X_test)
        results.append({**row_base, 'model': 'Transformer',
                         'acc': accuracy_score(y_test, pred_tfm),
                         'f1':  f1_score(y_test, pred_tfm)})

    return results


all_dl_results = []
for coin in COINS:
    print(f'{coin}...', end=' ', flush=True)
    coin_res = walk_forward_dl(coin)
    all_dl_results.extend(coin_res)
    n_w = len([r for r in coin_res if r['model'] == 'LSTM'])
    print(f'{n_w}w', end='  ', flush=True)

df_dl = pd.DataFrame(all_dl_results)
df_dl.to_csv('data/dl_baseline_results.csv', index=False)
print(f'\n\nSaved {len(df_dl)} rows → data/dl_baseline_results.csv')
print('\nMean accuracy by coin and model:')
print(df_dl.groupby(['coin','model'])['acc'].mean().unstack().round(4).to_string())

In [ ]:
# ── DM test: Best Ordinal vs LSTM / Transformer ───────────────────────────────
df_dl  = pd.read_csv('data/dl_baseline_results.csv')
wf     = pd.read_csv('data/walkforward_results.csv')

def dm_test_windows(errors1, errors2):
    d = np.asarray(errors1)**2 - np.asarray(errors2)**2
    n = len(d)
    if n < 6: return np.nan, np.nan
    d_mean = d.mean()
    gamma0 = np.var(d, ddof=1)
    gamma1 = np.cov(d[1:], d[:-1])[0,1] if n > 2 else 0.0
    nw_var = gamma0 + 2*gamma1
    if nw_var <= 0: return 0.0, 1.0
    dm_stat = d_mean / np.sqrt(abs(nw_var)/n)
    p_val   = float(2*(1 - stats.norm.cdf(abs(dm_stat))))
    return round(float(dm_stat), 3), round(p_val, 4)

dm_rows = []
for coin in COINS:
    ord_sub    = wf[wf['coin'] == coin]
    best_model = ord_sub.groupby('model')['acc'].mean().idxmax()
    ord_err    = 1 - ord_sub[ord_sub['model'] == best_model]['acc'].values

    for bl_name in ['LSTM', 'Transformer']:
        bl_err = 1 - df_dl[(df_dl['coin']==coin) & (df_dl['model']==bl_name)]['acc'].values
        n_min  = min(len(ord_err), len(bl_err))
        if n_min < 6: continue
        dm_s, dm_p = dm_test_windows(ord_err[:n_min], bl_err[:n_min])
        dm_rows.append({'coin': coin, 'ordinal_model': best_model,
                         'baseline': bl_name, 'DM_stat': dm_s,
                         'p_value': dm_p, 'significant_p05': dm_p < 0.05 if dm_p==dm_p else False})

dm_df = pd.DataFrame(dm_rows)
dm_df.to_csv('data/dl_dm_results.csv', index=False)
print('=== DM Test: Best Ordinal vs Deep Learning Baselines ===')
print(dm_df.to_string(index=False))
n_sig_before = dm_df['significant_p05'].sum()
n_tests = len(dm_df)
alpha_bonf = 0.05 / n_tests if n_tests > 0 else 0.05
n_sig_after = (dm_df['p_value'] < alpha_bonf).sum()
print(f'\nSignificant before Bonferroni: {n_sig_before}/{n_tests}')
print(f'Significant after  Bonferroni (α={alpha_bonf:.4f}): {n_sig_after}/{n_tests}')

In [ ]:
# ── Full comparison figure including DL models ───────────────────────────────
df_dl   = pd.read_csv('data/dl_baseline_results.csv')
df_enh  = pd.read_csv('data/enhanced_baseline_results.csv')
wf      = pd.read_csv('data/walkforward_results.csv')
bl      = pd.read_csv('data/baseline_results.csv')

wf_best = (wf.groupby(['coin','model'])['acc'].mean()
              .reset_index()
              .sort_values('acc', ascending=False)
              .groupby('coin').first().reset_index())
wf_best['label'] = 'Best Ordinal'

arima_m = bl.groupby('coin')['acc'].mean().reset_index()
arima_m['label'] = 'ARIMA(1,0,1)'

label_map = {'GARCH11_directional': 'GARCH(1,1)',
             'Naive_Persistence':   'Persistence',
             'Naive_Majority':      'Majority',
             'Random_Walk':         'Rand. Walk',
             'LSTM':                'LSTM',
             'Transformer':         'Transformer'}
enh_m = df_enh.groupby(['coin','model'])['acc'].mean().reset_index()
enh_m['label'] = enh_m['model'].map(label_map)
dl_m  = df_dl.groupby(['coin','model'])['acc'].mean().reset_index()
dl_m['label']  = dl_m['model'].map(label_map)

plot_df = pd.concat([
    wf_best[['coin','acc','label']],
    arima_m[['coin','acc','label']],
    enh_m[['coin','acc','label']],
    dl_m[['coin','acc','label']]
])

ORDER  = ['Best Ordinal','ARIMA(1,0,1)','GARCH(1,1)',
          'Persistence','Majority','Rand. Walk','LSTM','Transformer']
COLORS = ['#1565C0','#E65100','#6A1B9A','#2E7D32','#C62828','#757575','#00838F','#AD1457']

sns.set_style('whitegrid')
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, coin in enumerate(COINS):
    ax  = axes[i]
    sub = (plot_df[plot_df['coin'] == coin]
           .drop_duplicates('label')
           .set_index('label')['acc']
           .reindex(ORDER))
    bars = ax.bar(range(len(ORDER)), sub.values, color=COLORS, width=0.7, edgecolor='white')
    ax.axhline(0.5, color='black', linestyle='--', linewidth=1)
    ax.set_title(coin, fontsize=12, fontweight='bold')
    ax.set_xticks(range(len(ORDER)))
    ax.set_xticklabels(ORDER, rotation=45, ha='right', fontsize=7)
    lo = max(0.38, sub.dropna().min() - 0.03)
    hi = min(0.65, sub.dropna().max() + 0.05)
    ax.set_ylim(lo, hi)
    if i % 4 == 0:
        ax.set_ylabel('Mean Accuracy', fontsize=9)
    for bar, val in zip(bars, sub.values):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=6)

fig.suptitle(
    'Walk-Forward Directional Accuracy: Ordinal Network vs. All Baselines incl. LSTM & Transformer\n'
    '(Mean over rolling 30-day test windows, Apr 2021 – Apr 2026)',
    fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('data/fig_dl_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved → data/fig_dl_comparison.png')

In [ ]:
# ── Summary table (all models) ────────────────────────────────────────────────
df_dl   = pd.read_csv('data/dl_baseline_results.csv')
df_enh  = pd.read_csv('data/enhanced_baseline_results.csv')
wf      = pd.read_csv('data/walkforward_results.csv')
bl      = pd.read_csv('data/baseline_results.csv')

print('=== COMPLETE COMPARISON TABLE (mean walk-forward accuracy) ===')
header = 'Coin | Best Ordinal | ARIMA | GARCH | Persist | Majority | Rand.Walk | LSTM | Transf.'
print(header)
print('-' * len(header))

for coin in COINS:
    ord_acc  = wf[wf['coin']==coin].groupby('model')['acc'].mean().max()
    arima    = bl[bl['coin']==coin]['acc'].mean()
    garch    = df_enh[(df_enh['coin']==coin)&(df_enh['model']=='GARCH11_directional')]['acc'].mean()
    pers     = df_enh[(df_enh['coin']==coin)&(df_enh['model']=='Naive_Persistence')]['acc'].mean()
    maj      = df_enh[(df_enh['coin']==coin)&(df_enh['model']=='Naive_Majority')]['acc'].mean()
    rw       = df_enh[(df_enh['coin']==coin)&(df_enh['model']=='Random_Walk')]['acc'].mean()
    lstm_acc = df_dl[(df_dl['coin']==coin)&(df_dl['model']=='LSTM')]['acc'].mean()
    tfm_acc  = df_dl[(df_dl['coin']==coin)&(df_dl['model']=='Transformer')]['acc'].mean()
    print(f'{coin:5s} | {ord_acc:.4f}      | {arima:.4f} | {garch:.4f} | '
          f'{pers:.4f}  | {maj:.4f}   | {rw:.4f}   | {lstm_acc:.4f} | {tfm_acc:.4f}')

print('\n✓ All models cluster near 50%, confirming no predictability at daily frequency.')